# Phase 2 — EW-MOGA Main Experiment
Reconstructs the original 15 runs (3 seeds x 5 folds).

In [2]:
import sys
print(sys.executable)

e:\project1\.venv\Scripts\python.exe


In [3]:
import pandas as pd
import numpy as np
import shap
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from itertools import combinations
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
from scipy.stats import spearmanr
from deap import base, creator, tools, algorithms
print('All imports successful')

e:\project1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful


In [4]:
import sys
print(sys.executable)
print(sys.version)

e:\project1\.venv\Scripts\python.exe
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [5]:
BASE    = Path('E:/project1/EW_MOGA_Project')
DATA    = BASE / 'data' / 'Brain Tumor.csv'
RESULTS = BASE / 'results'
RESULTS.mkdir(exist_ok=True)

FEATURE_COLS = ['Mean','Variance','Standard Deviation','Entropy',
    'Skewness','Kurtosis','Contrast','Energy',
    'ASM','Homogeneity','Dissimilarity','Correlation','Coarseness']

N_FEATURES  = 13
POP_SIZE    = 100
N_GEN       = 20
CX_PROB     = 0.8
MUT_PROB    = 1.0 / N_FEATURES
N_TREES_SEARCH  = 30
N_TREES_FINAL   = 100
N_FOLDS     = 5
SEEDS       = [0, 1, 2]
N_PERTURB   = 5
B_BOOTSTRAP = 2
N_ABLATION_SAMPLES = 10
TOP_K       = 5

print('Configuration set')
print(f'Population: {POP_SIZE} | Generations: {N_GEN} | Seeds: {SEEDS}')

Configuration set
Population: 100 | Generations: 20 | Seeds: [0, 1, 2]


In [6]:
df = pd.read_csv(DATA)
X_raw = df[FEATURE_COLS].values.astype(float)
y_raw = df['Class'].values.astype(int)
print(f'Dataset loaded: {X_raw.shape}')
print(f'Class 0: {(y_raw==0).sum()} | Class 1: {(y_raw==1).sum()}')

Dataset loaded: (3762, 13)
Class 0: 2079 | Class 1: 1683


In [7]:
# ── Preprocessing helpers ────────────────────────────────────
def preprocess(X_tr, X_val, y_tr, random_state=42):
    scaler = MinMaxScaler()
    X_tr_sc  = scaler.fit_transform(X_tr)
    X_val_sc = scaler.transform(X_val)
    sm = SMOTE(random_state=random_state)
    X_tr_sm, y_tr_sm = sm.fit_resample(X_tr_sc, y_tr)
    return X_tr_sm, X_val_sc, y_tr_sm

# ── Train RF on selected features ────────────────────────────
def train_rf(X_tr, y_tr, mask, n_trees=N_TREES_SEARCH, seed=42):
    idx = [i for i,v in enumerate(mask) if v==1]
    if not idx:
        idx = [0]
    rf = RandomForestClassifier(n_estimators=n_trees,
                                random_state=seed, n_jobs=-1)
    rf.fit(X_tr[:, idx], y_tr)
    return rf, idx

print('Preprocessing functions defined')

Preprocessing functions defined


In [8]:
# ── SCI — SHAP Consistency Index ─────────────────────────────
def compute_sci(X_tr, y_tr, mask, seed=42):
    idx = [i for i,v in enumerate(mask) if v==1]
    if not idx:
        return 0.0
    rhos = []
    rankings = []
    rng = np.random.RandomState(seed)
    for _ in range(B_BOOTSTRAP):
        boot_idx = rng.choice(len(X_tr), len(X_tr), replace=True)
        X_b, y_b = X_tr[boot_idx], y_tr[boot_idx]
        rf = RandomForestClassifier(n_estimators=N_TREES_SEARCH,
                                    random_state=seed, n_jobs=-1)
        rf.fit(X_b[:, idx], y_b)
        bg = X_b[rng.choice(len(X_b), min(50,len(X_b)), replace=False)]
        exp = shap.TreeExplainer(rf, bg)
        sv  = exp.shap_values(X_b[:min(30,len(X_b)), :], check_additivity=False)
        if isinstance(sv, list):
            sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
        else:
            sv_arr = np.abs(sv)
        if sv_arr.ndim == 3:
            sv_arr = sv_arr.mean(axis=2)
        rankings.append(sv_arr.mean(axis=0))
    for i, j in combinations(range(len(rankings)), 2):
        rho, _ = spearmanr(rankings[i], rankings[j])
        rhos.append(rho if not np.isnan(rho) else 0.0)
    return float(np.mean(rhos)) if rhos else 0.0

# ── FID — Feature Importance Deviation ───────────────────────
_prev_ranking = [None]
def compute_fid(X_tr, y_tr, mask, seed=42):
    idx = [i for i,v in enumerate(mask) if v==1]
    if not idx:
        return 1.0
    rf = RandomForestClassifier(n_estimators=N_TREES_SEARCH,
                                random_state=seed, n_jobs=-1)
    rf.fit(X_tr[:, idx], y_tr)
    rng = np.random.RandomState(seed)
    bg  = X_tr[rng.choice(len(X_tr), min(50,len(X_tr)), replace=False)]
    exp = shap.TreeExplainer(rf, bg)
    sv  = exp.shap_values(X_tr[:min(30,len(X_tr)),:], check_additivity=False)
    if isinstance(sv, list):
        sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
    else:
        sv_arr = np.abs(sv)
    if sv_arr.ndim == 3:
        sv_arr = sv_arr.mean(axis=2)
    curr_rank = np.argsort(-sv_arr.mean(axis=0))
    k = min(TOP_K, len(idx))
    curr_top  = set(curr_rank[:k])
    if _prev_ranking[0] is None:
        _prev_ranking[0] = curr_top
        return 0.0
    prev_top = _prev_ranking[0]
    _prev_ranking[0] = curr_top
    inter = len(curr_top & prev_top)
    union = len(curr_top | prev_top)
    return 1.0 - (inter/union) if union > 0 else 0.0

# ── LFS — Local Faithfulness Score ───────────────────────────
def compute_lfs(rf, X_val, idx, seed=42):
    if len(X_val) == 0:
        return 0.0
    rng = np.random.RandomState(seed)
    sample_idx = rng.choice(len(X_val), min(N_ABLATION_SAMPLES, len(X_val)), replace=False)
    X_s = X_val[sample_idx][:, idx]
    bg  = X_s[:min(20,len(X_s))]
    exp = shap.TreeExplainer(rf, bg)
    sv  = exp.shap_values(X_s, check_additivity=False)
    if isinstance(sv, list):
        sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
    else:
        sv_arr = np.abs(sv)
    if sv_arr.ndim == 3:
        sv_arr = sv_arr.mean(axis=2)
    importance_order = np.argsort(-sv_arr.mean(axis=0))
    base_proba = rf.predict_proba(X_s)
    drops = []
    X_ablated = X_s.copy()
    for feat_idx in importance_order:
        X_ablated[:, feat_idx] = 0.0
        new_proba = rf.predict_proba(X_ablated)
        drop = np.mean(np.abs(base_proba - new_proba))
        drops.append(drop)
    return float(np.clip(np.mean(drops)*10, 0, 1))

print('SCI, FID, LFS functions defined')

SCI, FID, LFS functions defined


In [9]:
# ── Composite E(S) ───────────────────────────────────────────
def compute_es(X_tr, y_tr, X_val, mask, rf, idx,
               use_sci=True, use_fid=True, use_lfs=True, seed=42):
    components = []
    if use_sci:
        sci = compute_sci(X_tr, y_tr, mask, seed)
        components.append(sci)
    if use_fid:
        fid = compute_fid(X_tr, y_tr, mask, seed)
        components.append(1.0 - fid)
    if use_lfs:
        lfs = compute_lfs(rf, X_val, idx, seed)
        components.append(lfs)
    return float(np.mean(components)) if components else 0.0

print('E(S) composite function defined')

E(S) composite function defined


In [10]:
# ── Fitness evaluation ───────────────────────────────────────
def evaluate(individual, X_tr, y_tr, X_val, y_val,
             use_sci=True, use_fid=True, use_lfs=True, seed=42):
    mask = list(individual)
    idx  = [i for i,v in enumerate(mask) if v==1]
    if not idx:
        idx = [0]
        for i in range(len(mask)):
            mask[i] = 0
        mask[0] = 1
    rf, idx = train_rf(X_tr, y_tr, mask, n_trees=N_TREES_SEARCH, seed=seed)
    acc = accuracy_score(y_val, rf.predict(X_val[:, idx]))
    es  = compute_es(X_tr, y_tr, X_val, mask, rf, idx,
                     use_sci, use_fid, use_lfs, seed)
    return acc, es

print('Fitness evaluation function defined')

Fitness evaluation function defined


In [11]:
# ── NSGA-II setup ────────────────────────────────────────────
if 'FitnessMulti' in dir(creator):
    del creator.FitnessMulti
if 'Individual' in dir(creator):
    del creator.Individual

creator.create('FitnessMulti', base.Fitness, weights=(1.0, 1.0))
creator.create('Individual', list, fitness=creator.FitnessMulti)

def make_individual():
    ind = [np.random.randint(0,2) for _ in range(N_FEATURES)]
    if sum(ind) == 0:
        ind[np.random.randint(N_FEATURES)] = 1
    return creator.Individual(ind)

def mut_flip(individual, indpb):
    for i in range(len(individual)):
        if np.random.random() < indpb:
            individual[i] = 1 - individual[i]
    if sum(individual) == 0:
        individual[np.random.randint(N_FEATURES)] = 1
    return individual,

toolbox = base.Toolbox()
toolbox.register('individual', make_individual)
toolbox.register('population', tools.initRepeat, list, toolbox.individual)
toolbox.register('mate',    tools.cxTwoPoint)
toolbox.register('mutate',  mut_flip, indpb=MUT_PROB)
toolbox.register('select',  tools.selNSGA2)

print('NSGA-II toolbox configured')

NSGA-II toolbox configured


In [12]:
# ── CCC — Cross-Classifier Consistency ───────────────────────
def compute_ccc(X_tr, y_tr, X_val, idx, seed=42):
    from sklearn.svm import SVC
    from sklearn.neural_network import MLPClassifier
    import xgboost as xgb
    classifiers = {
        'RF' : RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1),
        'XGB': xgb.XGBClassifier(n_estimators=100, random_state=seed,
                                  eval_metric='logloss', verbosity=0),
        'SVM': SVC(kernel='rbf', probability=True, random_state=seed),
        'MLP': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300,
                             random_state=seed),
    }
    X_tr_sel  = X_tr[:, idx]
    X_val_sel = X_val[:, idx]
    rankings  = []
    rng = np.random.RandomState(seed)
    for name, clf in classifiers.items():
        try:
            clf.fit(X_tr_sel, y_tr)
            if name in ['RF','XGB']:
                exp = shap.TreeExplainer(clf)
            else:
                bg  = X_tr_sel[rng.choice(len(X_tr_sel), min(50,len(X_tr_sel)), replace=False)]
                exp = shap.KernelExplainer(clf.predict_proba, bg)
            sv  = exp.shap_values(X_val_sel[:min(30,len(X_val_sel))],
                                   check_additivity=False)
            if isinstance(sv, list):
                sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
            else:
                sv_arr = np.abs(sv)
            if sv_arr.ndim == 3:
                sv_arr = sv_arr.mean(axis=2)
            rankings.append(sv_arr.mean(axis=0))
        except Exception as e:
            print(f'  CCC warning ({name}): {e}')
    if len(rankings) < 2:
        return 0.0
    rhos = []
    for i, j in combinations(range(len(rankings)), 2):
        rho, _ = spearmanr(rankings[i], rankings[j])
        rhos.append(rho if not np.isnan(rho) else 0.0)
    return float(np.mean(rhos))

# ── PR — Perturbation Robustness ─────────────────────────────
def compute_pr(X_tr, y_tr, idx, seed=42):
    rf_base = RandomForestClassifier(n_estimators=100,
                                     random_state=seed, n_jobs=-1)
    rf_base.fit(X_tr[:, idx], y_tr)
    rng = np.random.RandomState(seed)
    bg  = X_tr[rng.choice(len(X_tr), min(50,len(X_tr)), replace=False)]
    exp = shap.TreeExplainer(rf_base, bg)
    sv  = exp.shap_values(X_tr[:min(30,len(X_tr)),:], check_additivity=False)
    if isinstance(sv, list):
        sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
    else:
        sv_arr = np.abs(sv)
    if sv_arr.ndim == 3:
        sv_arr = sv_arr.mean(axis=2)
    base_rank = sv_arr.mean(axis=0)
    rhos = []
    for p in range(N_PERTURB):
        boot_idx = rng.choice(len(X_tr), len(X_tr), replace=True)
        X_b, y_b = X_tr[boot_idx], y_tr[boot_idx]
        rf_p = RandomForestClassifier(n_estimators=100,
                                      random_state=seed+p, n_jobs=-1)
        rf_p.fit(X_b[:, idx], y_b)
        bg_p = X_b[rng.choice(len(X_b), min(50,len(X_b)), replace=False)]
        exp_p = shap.TreeExplainer(rf_p, bg_p)
        sv_p  = exp_p.shap_values(X_b[:min(30,len(X_b)),:], check_additivity=False)
        if isinstance(sv_p, list):
            sv_p_arr = np.mean([np.abs(r) for r in sv_p], axis=0)
        else:
            sv_p_arr = np.abs(sv_p)
        if sv_p_arr.ndim == 3:
            sv_p_arr = sv_p_arr.mean(axis=2)
        pert_rank = sv_p_arr.mean(axis=0)
        rho, _ = spearmanr(base_rank, pert_rank)
        rhos.append(rho if not np.isnan(rho) else 0.0)
    return float(np.mean(rhos))

print('CCC and PR audit functions defined')

CCC and PR audit functions defined


In [13]:
# ── Main EW-MOGA run function ────────────────────────────────
def run_ewmoga(X_tr, y_tr, X_val, y_val,
               use_sci=True, use_fid=True, use_lfs=True, seed=42):
    np.random.seed(seed)
    _prev_ranking[0] = None

    def eval_ind(ind):
        return evaluate(ind, X_tr, y_tr, X_val, y_val,
                        use_sci, use_fid, use_lfs, seed)

    toolbox.register('evaluate', eval_ind)
    pop = toolbox.population(n=POP_SIZE)

    # Evaluate initial population
    fits = list(map(toolbox.evaluate, pop))
    for ind, fit in zip(pop, fits):
        ind.fitness.values = fit

    pop = toolbox.select(pop, len(pop))

    for gen in range(N_GEN):
        offspring = tools.selTournamentDCD(pop, len(pop))
        offspring = [toolbox.clone(o) for o in offspring]
        for c1, c2 in zip(offspring[::2], offspring[1::2]):
            if np.random.random() < CX_PROB:
                toolbox.mate(c1, c2)
                del c1.fitness.values
                del c2.fitness.values
        for mut in offspring:
            toolbox.mutate(mut)
            if not mut.fitness.valid:
                del mut.fitness.values
        invalid = [ind for ind in offspring if not ind.fitness.valid]
        fits = list(map(toolbox.evaluate, invalid))
        for ind, fit in zip(invalid, fits):
            ind.fitness.values = fit
        pop = toolbox.select(pop + offspring, POP_SIZE)

    # Select best from Pareto front
    pareto = tools.sortNondominated(pop, len(pop), first_front_only=True)[0]
    best   = max(pareto, key=lambda ind: sum(ind.fitness.values))
    mask   = list(best)
    idx    = [i for i,v in enumerate(mask) if v==1]
    if not idx:
        idx = [0]

    # Final RF with 100 trees
    rf_final = RandomForestClassifier(n_estimators=N_TREES_FINAL,
                                       random_state=seed, n_jobs=-1)
    rf_final.fit(X_tr[:, idx], y_tr)
    final_acc = accuracy_score(y_val, rf_final.predict(X_val[:, idx]))
    es_val    = best.fitness.values[1]
    feat_names = ', '.join([FEATURE_COLS[i] for i in idx])

    # Post-hoc audit
    ccc = compute_ccc(X_tr, y_tr, X_val, idx, seed)
    pr  = compute_pr(X_tr, y_tr, idx, seed)

    return {
        'accuracy'             : final_acc,
        'n_features'           : len(idx),
        'feature_names'        : feat_names,
        'in_loop_explainability': es_val,
        'audit_ccc'            : ccc,
        'audit_pr'             : pr,
    }

print('EW-MOGA run function defined')

EW-MOGA run function defined


In [15]:
# ── Run all 15 experiments ───────────────────────────────────
import os

CHECKPOINT_FILE = RESULTS / 'ew_moga_checkpoint.csv'

# ── Resume from checkpoint if it exists ──────────────────────
if CHECKPOINT_FILE.exists():
    df_existing = pd.read_csv(CHECKPOINT_FILE)
    results = df_existing.to_dict('records')
    completed = set(zip(df_existing['seed'], df_existing['fold']))
    print(f'Resuming from checkpoint — {len(results)} runs already done')
else:
    results = []
    completed = set()
    print('Starting fresh — no checkpoint found')

total   = len(SEEDS) * N_FOLDS
run_num = len(results)

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_raw, y_raw)):

        # ── Skip already completed runs ───────────────────────
        if (seed, fold) in completed:
            print(f'[SKIP] seed={seed} fold={fold} already in checkpoint')
            continue

        run_num += 1
        print(f'[{run_num}/{total}] seed={seed} fold={fold}', end=' ... ')
        t0 = time.time()

        X_tr, X_val = X_raw[tr_idx], X_raw[val_idx]
        y_tr, y_val = y_raw[tr_idx], y_raw[val_idx]
        X_tr_pp, X_val_pp, y_tr_pp = preprocess(X_tr, X_val, y_tr, seed)

        res = run_ewmoga(X_tr_pp, y_tr_pp, X_val_pp, y_val, seed=seed)
        elapsed = time.time() - t0

        results.append({
            'method'                : 'EW_MOGA',
            'seed'                  : seed,
            'fold'                  : fold,
            'accuracy'              : res['accuracy'],
            'n_features'            : res['n_features'],
            'runtime_sec'           : elapsed,
            'feature_names'         : res['feature_names'],
            'in_loop_explainability': res['in_loop_explainability'],
            'audit_ccc'             : res['audit_ccc'],
            'audit_pr'              : res['audit_pr'],
        })

        # ── Save checkpoint immediately after every run ───────
        df_checkpoint = pd.DataFrame(results)
        df_checkpoint.to_csv(CHECKPOINT_FILE, index=False)
        print(f'acc={res["accuracy"]:.4f} ccc={res["audit_ccc"]:.3f} '
              f'pr={res["audit_pr"]:.3f} [{elapsed:.0f}s] ✓ saved')

        completed.add((seed, fold))

df_res = pd.DataFrame(results)
print(f'\nAll {len(df_res)} runs complete')
print(f'Checkpoint saved at: {CHECKPOINT_FILE}')

Starting fresh — no checkpoint found
[1/15] seed=0 fold=0 ... 

100%|██████████| 30/30 [00:00<00:00, 141.89it/s]


acc=0.9894 ccc=0.629 pr=0.985 [774s] ✓ saved
[2/15] seed=0 fold=1 ... 

100%|██████████| 30/30 [00:00<00:00, 387.31it/s]


acc=0.9907 ccc=0.550 pr=0.947 [773s] ✓ saved
[3/15] seed=0 fold=2 ... 

100%|██████████| 30/30 [00:00<00:00, 123.30it/s]


... (21 outputs left out)

e:\project1\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
e:\project1\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
e:\project1\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
e:\project1\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should b

acc=0.9920 ccc=0.631 pr=0.937 [560s] ✓ saved
[15/15] seed=2 fold=4 ... 

e:\project1\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
100%|██████████| 30/30 [00:00<00:00, 444.33it/s]


acc=0.9880 ccc=1.000 pr=0.998 [554s] ✓ saved

All 15 runs complete
Checkpoint saved at: E:\project1\EW_MOGA_Project\results\ew_moga_checkpoint.csv


In [16]:
# ── Save results ─────────────────────────────────────────────
out_path = RESULTS / 'ew_moga_results.csv'
df_res.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print()
print('='*55)
print('EW-MOGA SUMMARY')
print('='*55)
print(f'Accuracy  : {df_res.accuracy.mean():.4f} ± {df_res.accuracy.std():.4f}')
print(f'N features: {df_res.n_features.mean():.2f} ± {df_res.n_features.std():.2f}')
print(f'CCC       : {df_res.audit_ccc.mean():.4f} ± {df_res.audit_ccc.std():.4f}')
print(f'PR        : {df_res.audit_pr.mean():.4f} ± {df_res.audit_pr.std():.4f}')
print(f'Runtime   : {df_res.runtime_sec.mean():.1f}s mean')
print()
print('Notebook 2 complete — ready to run Notebook 3')
print('='*55)

Saved: E:\project1\EW_MOGA_Project\results\ew_moga_results.csv

EW-MOGA SUMMARY
Accuracy  : 0.9902 ± 0.0035
N features: 5.73 ± 1.33
CCC       : 0.6210 ± 0.1451
PR        : 0.9651 ± 0.0324
Runtime   : 742.8s mean

Notebook 2 complete — ready to run Notebook 3
